In [1]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Cấu hình đường dẫn dữ liệu
DATA_DIR = "/data/raw"
MAIN_TRAIN_FILE = os.path.join(DATA_DIR, "application_train.csv")

# Nạp dữ liệu thô
df_train = pd.read_csv(MAIN_TRAIN_FILE)
print(f"✅ Đã nạp thành công dữ liệu thô: {df_train.shape[0]:,} dòng | {df_train.shape[1]} cột.")

✅ Đã nạp thành công dữ liệu thô: 307,511 dòng | 122 cột.


In [2]:
# Danh sách 23 cột Keep 100% đã được thống nhất sau EDA
KEEP_COLUMNS_100 = [
    'SK_ID_CURR', 'TARGET', 'CODE_GENDER', 'CNT_CHILDREN',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS',
    'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
    'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE'
]

def filter_keep_features(df: pd.DataFrame, feature_list: list) -> pd.DataFrame:
    """
    Hàm lọc và giữ lại đúng danh sách các thuộc tính đã qua thẩm định từ bước EDA.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame thô ban đầu.
    feature_list : list
        Danh sách tên các cột được giữ lại ('Keep 100%').

    Returns:
    --------
    pd.DataFrame
        DataFrame đã được làm sạch và thu gọn cột.
    """
    # 1. Kiểm tra các cột thực tế có tồn tại trong DataFrame không
    valid_cols = [col for col in feature_list if col in df.columns]
    missing_cols = set(feature_list) - set(valid_cols)

    if missing_cols:
        print(f"⚠️ Cảnh báo: Có {len(missing_cols)} cột không tìm thấy trong file input: {missing_cols}")

    # 2. Tiến hành lọc cột
    df_filtered = df[valid_cols].copy()

    # 3. Thống kê thông số tối ưu bộ nhớ
    orig_cols = df.shape[1]
    new_cols = df_filtered.shape[1]
    mem_before = df.memory_usage().sum() / 1024**2
    mem_after = df_filtered.memory_usage().sum() / 1024**2

    print("🧹 [FEATURE SELECTION LOG]")
    print(f"  • Số cột ban đầu: {orig_cols} ➔ Số cột giữ lại: {new_cols} (Đã loại bỏ {orig_cols - new_cols} cột nhiễu)")
    print(f"  • Dung lượng RAM: {mem_before:.2f} MB ➔ {mem_after:.2f} MB (Tiết kiệm {((mem_before - mem_after)/mem_before)*100:.1f}% bộ nhớ)")

    return df_filtered


In [3]:
# Gọi hàm lọc dữ liệu cho bảng chính application_train
df_main_clean = filter_keep_features(df_train, KEEP_COLUMNS_100)

# Hiển thị 5 dòng đầu tiên của bảng đã lọc
display(df_main_clean.head())

🧹 [FEATURE SELECTION LOG]
  • Số cột ban đầu: 122 ➔ Số cột giữ lại: 23 (Đã loại bỏ 99 cột nhiễu)
  • Dung lượng RAM: 286.23 MB ➔ 53.96 MB (Tiết kiệm 81.1% bộ nhớ)


,SK_ID_CURR,TARGET,CODE_GENDER,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,...,OCCUPATION_TYPE,CNT_FAM_MEMBERS,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE
0,100002,1,M,0,202500.0,406597.5,24700.5,351000.0,Working,Secondary / secondary special,...,Laborers,1.0,Business Entity Type 3,0.083037,0.262949,0.139376,2.0,2.0,2.0,2.0
1,100003,0,F,0,270000.0,1293502.5,35698.5,1129500.0,State servant,Higher education,...,Core staff,2.0,School,0.311267,0.622246,NaN,1.0,0.0,1.0,0.0
2,100004,0,M,0,67500.0,135000.0,6750.0,135000.0,Working,Secondary / secondary special,...,Laborers,1.0,Government,NaN,0.555912,0.729567,0.0,0.0,0.0,0.0
3,100006,0,F,0,135000.0,312682.5,29686.5,297000.0,Working,Secondary / secondary special,...,Laborers,2.0,Business Entity Type 3,NaN,0.650442,NaN,2.0,0.0,2.0,0.0
4,100007,0,M,0,121500.0,513000.0,21865.5,513000.0,Working,Secondary / secondary special,...,Core staff,1.0,Religion,NaN,0.322738,NaN,0.0,0.0,0.0,0.0


In [5]:
# Replace the anomaly value 'XNA' in CODE_GENDER with the majority class ('F')
df_main_clean['CODE_GENDER'] = df_main_clean['CODE_GENDER'].replace('XNA', 'F')

# Log output verification
print(f"✅ CODE_GENDER processed successfully. Current 'F' ratio: {(df_main_clean['CODE_GENDER'] == 'F').mean() * 100:.2f}%")

✅ CODE_GENDER processed successfully. Current 'F' ratio: 65.84%


Replace 4 XNA with F

In [6]:
# 1. Cap CNT_CHILDREN at an upper threshold of 5
df_main_clean['CNT_CHILDREN'] = df_main_clean['CNT_CHILDREN'].clip(upper=5)

# 2. Quick verification log
print("✅ CNT_CHILDREN - Max value after capping:", df_main_clean['CNT_CHILDREN'].max())
print("✅ CNT_CHILDREN - Distribution count:")
print(df_main_clean['CNT_CHILDREN'].value_counts().sort_index())

✅ CNT_CHILDREN - Max value after capping: 5
✅ CNT_CHILDREN - Distribution count:
CNT_CHILDREN
0    215371
1     61119
2     26749
3      3717
4       429
5       126
Name: count, dtype: int64


### 📌 Preprocessing: `CNT_CHILDREN` (Count of Children)

- **Action Taken:** Applied upper-bound capping (clipping) at a threshold of 5 children.
- **Method Applied:** `df_main_clean['CNT_CHILDREN'] = df_main_clean['CNT_CHILDREN'].clip(upper=5)`
- **Rationale:**
  - Over 99.9% of applicants have between 0 and 5 children.
  - Extreme outliers (up to 19 children) distort downstream financial feature engineering ratios (e.g., `INCOME_PER_PERSON`).
  - Capping values at 5 preserves 100% of total rows while mitigating extreme skewness for gradient boosting models (LightGBM/XGBoost).

In [7]:
# 1. Calculate dynamic upper threshold at 99.9th percentile
income_cap = df_main_clean['AMT_INCOME_TOTAL'].quantile(0.999)

# 2. Apply upper-bound clipping
df_main_clean['AMT_INCOME_TOTAL'] = df_main_clean['AMT_INCOME_TOTAL'].clip(upper=income_cap)

# 3. Verification log
print(f"✅ AMT_INCOME_TOTAL - 99.9th Percentile Cap Threshold: {income_cap:,.2f}")
print(f"✅ AMT_INCOME_TOTAL - Max value after capping: {df_main_clean['AMT_INCOME_TOTAL'].max():,.2f}")
print(f"✅ AMT_INCOME_TOTAL - Mean value after capping: {df_main_clean['AMT_INCOME_TOTAL'].mean():,.2f}")

✅ AMT_INCOME_TOTAL - 99.9th Percentile Cap Threshold: 900,000.00
✅ AMT_INCOME_TOTAL - Max value after capping: 900,000.00
✅ AMT_INCOME_TOTAL - Mean value after capping: 167,771.35


### 📌 Preprocessing: `AMT_INCOME_TOTAL` (Total Applicant Income)

- **Action Taken:** Applied upper-bound quantile capping at the 99.9th percentile.
- **Method Applied:**
  `income_cap = df_main_clean['AMT_INCOME_TOTAL'].quantile(0.999)`
  `df_main_clean['AMT_INCOME_TOTAL'] = df_main_clean['AMT_INCOME_TOTAL'].clip(upper=income_cap)`
- **Rationale:**
  - An extreme data entry anomaly exists where max income reaches 117,000,000 (over 700x the median of 147,150).
  - This extreme outlier heavily skews statistical distributions and corrupts key financial ratios like `CREDIT_TO_INCOME_RATIO`.
  - Dynamic quantile capping at 99.9% neutralizes the anomaly while preserving high-income variance and 100% of row entries.

In [9]:
# 1. Verify feature status (No structural modification needed)
null_count = df_main_clean['AMT_CREDIT'].isnull().sum()
min_val = df_main_clean['AMT_CREDIT'].min()
max_val = df_main_clean['AMT_CREDIT'].max()

# 2. Verification log
print(f"✅ AMT_CREDIT - Missing Values: {null_count}")
print(f"✅ AMT_CREDIT - Range: [{min_val:,.2f} ➔ {max_val:,.2f}]")
print(f"✅ AMT_CREDIT - Status: Clean continuous feature retained.")

✅ AMT_CREDIT - Missing Values: 0
✅ AMT_CREDIT - Range: [45,000.00 ➔ 4,050,000.00]
✅ AMT_CREDIT - Status: Clean continuous feature retained.


### 📌 Preprocessing: `AMT_CREDIT` (Credit Amount of the Loan)

- **Action Taken:** Inspected feature distribution; no capping or outlier filtering required.
- **Status:** Retained in original continuous numerical form.
- **Rationale:**
  - Zero missing values (0.00% missing rate).
  - Exhibits a smooth, naturally right-skewed distribution ranging from 45,000 to 4,050,000.
  - The maximum credit amount of 4.05M represents valid high-value loan products rather than data entry error.
  - Tree-based models (LightGBM/XGBoost) efficiently handle raw continuous distributions without requiring scaling or truncation.

In [11]:
# 1. Calculate median value for imputation
annuity_median = df_main_clean['AMT_ANNUITY'].median()

# 2. Impute missing values with median
missing_before = df_main_clean['AMT_ANNUITY'].isnull().sum()
df_main_clean['AMT_ANNUITY'] = df_main_clean['AMT_ANNUITY'].fillna(annuity_median)
missing_after = df_main_clean['AMT_ANNUITY'].isnull().sum()

# 3. Verification log
print(f"✅ AMT_ANNUITY - Imputed Median Value: {annuity_median:,.2f}")
print(f"✅ AMT_ANNUITY - Missing Values (Before ➔ After): {missing_before} ➔ {missing_after}")
print(f"✅ AMT_ANNUITY - Status: Missing records successfully resolved.")

✅ AMT_ANNUITY - Imputed Median Value: 24,903.00
✅ AMT_ANNUITY - Missing Values (Before ➔ After): 12 ➔ 0
✅ AMT_ANNUITY - Status: Missing records successfully resolved.


### 📌 Preprocessing: `AMT_ANNUITY` (Loan Annuity Amount)

- **Action Taken:** Imputed 12 missing values using the feature median; retained original continuous scale without truncation.
- **Method Applied:** `df_main_clean['AMT_ANNUITY'] = df_main_clean['AMT_ANNUITY'].fillna(df_main_clean['AMT_ANNUITY'].median())`
- **Rationale:**
  - Only 12 records (0.004%) are missing.
  - Imputing missing values with the median prevents downstream `NaN` propagation during financial feature engineering (e.g., `CREDIT_TO_ANNUITY_RATIO`).
  - The maximum value (258,025.50) follows a natural right-skewed distribution aligned with high `AMT_CREDIT` values, requiring no outlier capping.

In [12]:
# 1. Impute missing AMT_GOODS_PRICE using AMT_CREDIT (Domain-driven Imputation)
missing_before = df_main_clean['AMT_GOODS_PRICE'].isnull().sum()

df_main_clean['AMT_GOODS_PRICE'] = df_main_clean['AMT_GOODS_PRICE'].fillna(df_main_clean['AMT_CREDIT'])

missing_after = df_main_clean['AMT_GOODS_PRICE'].isnull().sum()

# 2. Verification log
print(f"✅ AMT_GOODS_PRICE - Missing Values (Before ➔ After): {missing_before} ➔ {missing_after}")
print(f"✅ AMT_GOODS_PRICE - Domain Imputation Completed: Filled using AMT_CREDIT values.")

✅ AMT_GOODS_PRICE - Missing Values (Before ➔ After): 278 ➔ 0
✅ AMT_GOODS_PRICE - Domain Imputation Completed: Filled using AMT_CREDIT values.


### 📌 Preprocessing: `AMT_GOODS_PRICE` (Price of Goods for Credit)

- **Action Taken:** Imputed 278 missing values dynamically using `AMT_CREDIT` instead of global median.
- **Method Applied:** `df_main_clean['AMT_GOODS_PRICE'] = df_main_clean['AMT_GOODS_PRICE'].fillna(df_main_clean['AMT_CREDIT'])`
- **Rationale:**
  - Aligns with banking domain logic: For cash loans without underlying physical goods, `AMT_GOODS_PRICE` equals the loan amount `AMT_CREDIT`.
  - Guarantees a logical baseline ratio of `GOODS_TO_CREDIT_RATIO = 1.0` for missing records during feature engineering.
  - Resolves all 278 missing values (0.09%) without distorting financial contract properties.

In [14]:
# 1. Define list of rare categories (frequency < 100 rows)
rare_income_types = ['Unemployed', 'Student', 'Businessman', 'Maternity leave']

# 2. Consolidate rare categories into 'Other'
df_main_clean['NAME_INCOME_TYPE'] = df_main_clean['NAME_INCOME_TYPE'].replace(rare_income_types, 'Other')

# 3. Verification log
print("✅ NAME_INCOME_TYPE - Value distribution after grouping rare categories:")
print(df_main_clean['NAME_INCOME_TYPE'].value_counts())

✅ NAME_INCOME_TYPE - Value distribution after grouping rare categories:
NAME_INCOME_TYPE
Working                 158774
Commercial associate     71617
Pensioner                55362
State servant            21703
Other                       55
Name: count, dtype: int64


### 📌 Preprocessing: `NAME_INCOME_TYPE` (Income Type of Applicant)

- **Action Taken:** Grouped rare income categories (`Unemployed`, `Student`, `Businessman`, `Maternity leave`) into a consolidated `'Other'` category.
- **Method Applied:**
  `rare_categories = ['Unemployed', 'Student', 'Businessman', 'Maternity leave']`
  `df_main_clean['NAME_INCOME_TYPE'] = df_main_clean['NAME_INCOME_TYPE'].replace(rare_categories, 'Other')`
- **Rationale:**
  - Zero missing values (0.00% missing rate).
  - Four rare categories account for only 55 out of 307,511 records (~0.018%).
  - Extremely rare classes introduce high variance, potential CV fold coverage issues (e.g., `Maternity leave` has only 5 samples), and encoding sparsity.
  - Consolidating rare classes simplifies the feature space to 5 robust categories (`Working`, `Commercial associate`, `Pensioner`, `State servant`, `Other`).

### 📌 Preprocessing: `NAME_EDUCATION_TYPE` (Education Level of Applicant)

- **Action Taken:** Inspected categorical distribution; retained all 5 original categories without modification or grouping.
- **Status:** Kept in original categorical form.
- **Rationale:**
  - Zero missing values (0.00% missing rate).
  - Possesses a natural ordinal hierarchy representing applicant education levels.
  - The minority class `Academic degree` contains 164 samples (~0.05%), providing a strong domain signal for creditworthiness with sufficient support across cross-validation folds.

In [17]:
# 1. Replace 'Unknown' categorical anomaly with majority class ('Married')
df_main_clean['NAME_FAMILY_STATUS'] = df_main_clean['NAME_FAMILY_STATUS'].replace('Unknown', 'Married')

# 2. Verification log
print("✅ NAME_FAMILY_STATUS - Category distribution after fixing 'Unknown':")
print(df_main_clean['NAME_FAMILY_STATUS'].value_counts())
print(f"✅ NAME_FAMILY_STATUS - Remaining 'Unknown' count: {(df_main_clean['NAME_FAMILY_STATUS'] == 'Unknown').sum()}")

✅ NAME_FAMILY_STATUS - Category distribution after fixing 'Unknown':
NAME_FAMILY_STATUS
Married                 196434
Single / not married     45444
Civil marriage           29775
Separated                19770
Widow                    16088
Name: count, dtype: int64
✅ NAME_FAMILY_STATUS - Remaining 'Unknown' count: 0


### 📌 Preprocessing: `NAME_FAMILY_STATUS` (Family / Marital Status)

- **Action Taken:** Replaced 2 anomalous `'Unknown'` records with the majority class (`'Married'`).
- **Method Applied:** `df_main_clean['NAME_FAMILY_STATUS'] = df_main_clean['NAME_FAMILY_STATUS'].replace('Unknown', 'Married')`
- **Rationale:**
  - Zero missing values (0.00% missing rate).
  - Contains an extreme categorical anomaly `'Unknown'` with only 2 records (~0.00%).
  - Mode imputation eliminates unnecessary encoding sparsity and ensures robust pipeline compatibility across training and test sets without dropping rows.

### 📌 Preprocessing: `DAYS_BIRTH` (Client Age in Days)

- **Action Taken:** Inspected feature distribution; verified data cleanliness with no preprocessing or capping required.
- **Status:** Retained in original continuous numerical form.
- **Rationale:**
  - Zero missing values (0.00% missing rate).
  - Negative integers represent days elapsed before loan application.
  - Converting days to years reveals an ideal working-age range from 20.5 to 69.1 years old (Mean: ~43.9 years).
  - Distribution is perfectly symmetric with zero statistical outliers on boxplot inspection.

In [18]:
import numpy as np

# 1. Count anomaly occurrences before replacement
anom_count = (df_main_clean['DAYS_EMPLOYED'] == 365243).sum()

# 2. Replace 365243 anomaly with NaN
df_main_clean['DAYS_EMPLOYED'] = df_main_clean['DAYS_EMPLOYED'].replace(365243, np.nan)

# 3. Verification log
print(f"✅ DAYS_EMPLOYED - Anomaly records (365243) converted to NaN: {anom_count:,}")
print(f"✅ DAYS_EMPLOYED - Valid employment range (in years): [{abs(df_main_clean['DAYS_EMPLOYED'].min())/365.25:.1f} years ➔ {abs(df_main_clean['DAYS_EMPLOYED'].max())/365.25:.1f} years]")
print(f"✅ DAYS_EMPLOYED - Current NaN count: {df_main_clean['DAYS_EMPLOYED'].isnull().sum():,}")

✅ DAYS_EMPLOYED - Anomaly records (365243) converted to NaN: 55,374
✅ DAYS_EMPLOYED - Valid employment range (in years): [49.0 years ➔ 0.0 years]
✅ DAYS_EMPLOYED - Current NaN count: 55,374


### 📌 Preprocessing: `DAYS_EMPLOYED` (Days of Employment Before Application)

- **Action Taken:** Replaced the famous anomaly value `365243` with `np.nan`.
- **Method Applied:** `df_main_clean['DAYS_EMPLOYED'] = df_main_clean['DAYS_EMPLOYED'].replace(365243, np.nan)`
- **Rationale:**
  - `365243` represents ~1,000 years of employment, a well-documented synthetic placeholder value used by Home Credit for pensioners and unemployed applicants.
  - Leaving `365243` intact severely distorts tree-based split decisions and downstream ratio features (e.g., `EMPLOYMENT_TO_AGE_RATIO`).
  - Replacing it with `np.nan` restores the true employment distribution (0 to ~49 years) while allowing LightGBM/XGBoost to handle missing values natively.

In [19]:
# 1. Fill NaN values with explicit 'Missing' category
missing_before = df_main_clean['OCCUPATION_TYPE'].isnull().sum()
df_main_clean['OCCUPATION_TYPE'] = df_main_clean['OCCUPATION_TYPE'].fillna('Missing')
missing_after = df_main_clean['OCCUPATION_TYPE'].isnull().sum()

# 2. Verification log
print(f"✅ OCCUPATION_TYPE - Missing Values Imputed (Before ➔ After): {missing_before:,} ➔ {missing_after}")
print("✅ OCCUPATION_TYPE - Top 5 Category distribution:")
print(df_main_clean['OCCUPATION_TYPE'].value_counts().head(5))

✅ OCCUPATION_TYPE - Missing Values Imputed (Before ➔ After): 96,391 ➔ 0
✅ OCCUPATION_TYPE - Top 5 Category distribution:
OCCUPATION_TYPE
Missing        96391
Laborers       55186
Sales staff    32102
Core staff     27570
Managers       21371
Name: count, dtype: int64


### 📌 Preprocessing: `OCCUPATION_TYPE` (Client Occupation Category)

- **Action Taken:** Explicitly imputed missing values by creating a dedicated `'Missing'` category.
- **Method Applied:** `df_main_clean['OCCUPATION_TYPE'] = df_main_clean['OCCUPATION_TYPE'].fillna('Missing')`
- **Rationale:**
  - High missing rate of 31.35% (96,391 rows).
  - Missingness heavily correlates with non-working applicants (e.g., pensioners/unemployed) and unrecorded occupations.
  - Converting `NaN` into an explicit `'Missing'` label preserves predictive signal, prevents data loss, and avoids artificial distribution bias caused by mode imputation.

In [20]:
# 1. Impute 2 missing values with median
fam_median = df_main_clean['CNT_FAM_MEMBERS'].median()
df_main_clean['CNT_FAM_MEMBERS'] = df_main_clean['CNT_FAM_MEMBERS'].fillna(fam_median)

# 2. Cap extreme outliers at upper threshold of 6
df_main_clean['CNT_FAM_MEMBERS'] = df_main_clean['CNT_FAM_MEMBERS'].clip(upper=6)

# 3. Verification log
print(f"✅ CNT_FAM_MEMBERS - Missing Values remaining: {df_main_clean['CNT_FAM_MEMBERS'].isnull().sum()}")
print(f"✅ CNT_FAM_MEMBERS - Max Value after capping: {df_main_clean['CNT_FAM_MEMBERS'].max()}")
print("✅ CNT_FAM_MEMBERS - Distribution count:")
print(df_main_clean['CNT_FAM_MEMBERS'].value_counts().sort_index())

✅ CNT_FAM_MEMBERS - Missing Values remaining: 0
✅ CNT_FAM_MEMBERS - Max Value after capping: 6.0
✅ CNT_FAM_MEMBERS - Distribution count:
CNT_FAM_MEMBERS
1.0     67847
2.0    158359
3.0     52601
4.0     24697
5.0      3478
6.0       529
Name: count, dtype: int64


### 📌 Preprocessing: `CNT_FAM_MEMBERS` (Count of Family Members)

- **Action Taken:** Imputed 2 missing values with feature median (2.0) and applied upper-bound capping at a threshold of 6 family members.
- **Method Applied:**
  `df_main_clean['CNT_FAM_MEMBERS'] = df_main_clean['CNT_FAM_MEMBERS'].fillna(df_main_clean['CNT_FAM_MEMBERS'].median())`
  `df_main_clean['CNT_FAM_MEMBERS'] = df_main_clean['CNT_FAM_MEMBERS'].clip(upper=6)`
- **Rationale:**
  - Resolves 2 missing values (0.00%) to maintain data completeness for downstream per-capita ratio engineering (e.g., `INCOME_PER_PERSON`).
  - Over 99% of households have between 1 and 5 members. Extreme outliers (up to 20 members) skew financial ratio distributions.
  - Capping at 6 maintains logical consistency with `CNT_CHILDREN` (capped at 5) while preserving full dataset row integrity.

In [21]:
# 1. Replace 'XNA' synthetic value with 'Missing'
df_main_clean['ORGANIZATION_TYPE'] = df_main_clean['ORGANIZATION_TYPE'].replace('XNA', 'Missing')

# 2. Identify rare categories with count < 1000
org_counts = df_main_clean['ORGANIZATION_TYPE'].value_counts()
rare_orgs = org_counts[org_counts < 1000].index.tolist()

# 3. Group rare categories into 'Other'
df_main_clean['ORGANIZATION_TYPE'] = df_main_clean['ORGANIZATION_TYPE'].replace(rare_orgs, 'Other')

# 4. Verification log
print(f"✅ ORGANIZATION_TYPE - Reduced categories from {len(org_counts)} ➔ {df_main_clean['ORGANIZATION_TYPE'].nunique()}")
print(f"✅ ORGANIZATION_TYPE - Total rare categories consolidated: {len(rare_orgs)}")
print("✅ ORGANIZATION_TYPE - Top 10 categories after processing:")
print(df_main_clean['ORGANIZATION_TYPE'].value_counts().head(10))

✅ ORGANIZATION_TYPE - Reduced categories from 58 ➔ 33
✅ ORGANIZATION_TYPE - Total rare categories consolidated: 25
✅ ORGANIZATION_TYPE - Top 10 categories after processing:
ORGANIZATION_TYPE
Business Entity Type 3    67992
Missing                   55374
Self-employed             38412
Other                     26412
Medicine                  11193
Business Entity Type 2    10553
Government                10404
School                     8893
Trade: type 7              7831
Kindergarten               6880
Name: count, dtype: int64


### 📌 Preprocessing: `ORGANIZATION_TYPE` (Type of Organization where Client Works)

- **Action Taken:** Handled synthetic missing value (`XNA`) and consolidated low-frequency rare organization types into `'Other'`.
- **Method Applied:**
  1. Replaced `'XNA'` with `'Missing'`.
  2. Grouped categories with frequency < 1,000 records into `'Other'`.
- **Rationale:**
  - High cardinality feature with 58 distinct categories.
  - `'XNA'` represents 55,374 non-working records (18.01%) and is explicitly relabeled to `'Missing'` to retain predictive signal.
  - Consolidating rare categories (<0.3% frequency) reduces encoding sparsity, speeds up GBDT training, and prevents overfitting on low-sample organization types (e.g., `Religion` with only 85 rows).

In [22]:
# 1. Inspect missingness and continuous metrics for EXT_SOURCE_1
ext1_null = df_main_clean['EXT_SOURCE_1'].isnull().sum()
ext1_null_pct = df_main_clean['EXT_SOURCE_1'].isnull().mean() * 100

# 2. Verification log
print(f"✅ EXT_SOURCE_1 - Missing Values: {ext1_null:,} ({ext1_null_pct:.2f}%)")
print(f"✅ EXT_SOURCE_1 - Valid Range: [{df_main_clean['EXT_SOURCE_1'].min():.2f} ➔ {df_main_clean['EXT_SOURCE_1'].max():.2f}]")
print("✅ EXT_SOURCE_1 - Status: Kept in raw form with NaNs intact for LightGBM/XGBoost native handling.")

✅ EXT_SOURCE_1 - Missing Values: 173,378 (56.38%)
✅ EXT_SOURCE_1 - Valid Range: [0.01 ➔ 0.96]
✅ EXT_SOURCE_1 - Status: Kept in raw form with NaNs intact for LightGBM/XGBoost native handling.


### 📌 Preprocessing: `EXT_SOURCE_1` (Normalized Score from External Data Source 1)

- **Action Taken:** Inspected feature quality and missingness; retained original scale with raw `np.nan` values preserved.
- **Status:** Retained as continuous feature without imputation or capping.
- **Rationale:**
  - High missing rate of 56.38% (173,378 rows) due to unrecorded external credit history.
  - `EXT_SOURCE_1` is one of the single most influential predictive features for credit default risk.
  - Imputing over 56% of rows with median/mean would severely distort its clean Gaussian distribution.
  - Tree-based algorithms (LightGBM/XGBoost) natively optimize split decisions for missing values without requiring artificial imputation.

In [23]:
# 1. Inspect missingness and continuous metrics for EXT_SOURCE_2
ext2_null = df_main_clean['EXT_SOURCE_2'].isnull().sum()
ext2_null_pct = df_main_clean['EXT_SOURCE_2'].isnull().mean() * 100

# 2. Verification log
print(f"✅ EXT_SOURCE_2 - Missing Values: {ext2_null:,} ({ext2_null_pct:.2f}%)")
print(f"✅ EXT_SOURCE_2 - Valid Range: [{df_main_clean['EXT_SOURCE_2'].min():.2f} ➔ {df_main_clean['EXT_SOURCE_2'].max():.2f}]")
print("✅ EXT_SOURCE_2 - Status: Primary predictive feature retained in raw continuous form.")

✅ EXT_SOURCE_2 - Missing Values: 660 (0.21%)
✅ EXT_SOURCE_2 - Valid Range: [0.00 ➔ 0.85]
✅ EXT_SOURCE_2 - Status: Primary predictive feature retained in raw continuous form.


### 📌 Preprocessing: `EXT_SOURCE_2` (Normalized Score from External Data Source 2)

- **Action Taken:** Inspected feature distribution and missingness; retained original continuous scale with raw values intact.
- **Status:** Retained as primary continuous feature without imputation or capping.
- **Rationale:**
  - Nearly complete feature with a negligible missing rate of 0.21% (660 rows).
  - Consistently ranks as the single most important predictive feature across gradient boosting baseline models.
  - Smooth, continuous distribution ranging from 0.00 to 0.85 with zero statistical outliers.
  - Preserving raw `NaN` values leverages native missing value handling in LightGBM/XGBoost.

In [24]:
# 1. Inspect missingness and continuous metrics for EXT_SOURCE_3
ext3_null = df_main_clean['EXT_SOURCE_3'].isnull().sum()
ext3_null_pct = df_main_clean['EXT_SOURCE_3'].isnull().mean() * 100

# 2. Verification log
print(f"✅ EXT_SOURCE_3 - Missing Values: {ext3_null:,} ({ext3_null_pct:.2f}%)")
print(f"✅ EXT_SOURCE_3 - Valid Range: [{df_main_clean['EXT_SOURCE_3'].min():.2f} ➔ {df_main_clean['EXT_SOURCE_3'].max():.2f}]")
print("✅ EXT_SOURCE_3 - Status: Top-tier predictive feature retained in raw continuous form.")

✅ EXT_SOURCE_3 - Missing Values: 60,965 (19.83%)
✅ EXT_SOURCE_3 - Valid Range: [0.00 ➔ 0.90]
✅ EXT_SOURCE_3 - Status: Top-tier predictive feature retained in raw continuous form.


### 📌 Preprocessing: `EXT_SOURCE_3` (Normalized Score from External Data Source 3)

- **Action Taken:** Inspected feature quality and distribution; retained raw continuous values and preserved `np.nan` entries.
- **Status:** Retained as primary continuous feature without imputation or capping.
- **Rationale:**
  - Moderate missing rate of 19.83% (60,965 rows).
  - Ranks as one of the top 3 most impactful predictive features for credit scoring models.
  - Displays a clean, smooth continuous distribution ranging from 0.00 to 0.90 with zero statistical outliers.
  - Preserving raw `np.nan` values allows LightGBM/XGBoost to natively handle missing splits during tree construction.

In [25]:
# 1. Impute missing values with median (0.0)
obs30_median = df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'].median()
df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'] = df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'].fillna(obs30_median)

# 2. Cap extreme outliers at upper threshold of 10
df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'] = df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'].clip(upper=10)

# 3. Verification log
print(f"✅ OBS_30_CNT_SOCIAL_CIRCLE - Remaining Missing Values: {df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'].isnull().sum()}")
print(f"✅ OBS_30_CNT_SOCIAL_CIRCLE - Max Value after capping: {df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'].max()}")
print("✅ OBS_30_CNT_SOCIAL_CIRCLE - Value counts summary:")
print(df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'].value_counts().sort_index())

✅ OBS_30_CNT_SOCIAL_CIRCLE - Remaining Missing Values: 0
✅ OBS_30_CNT_SOCIAL_CIRCLE - Max Value after capping: 10.0
✅ OBS_30_CNT_SOCIAL_CIRCLE - Value counts summary:
OBS_30_CNT_SOCIAL_CIRCLE
0.0     164931
1.0      48783
2.0      29808
3.0      20322
4.0      14143
5.0       9553
6.0       6453
7.0       4390
8.0       2967
9.0       2003
10.0      4158
Name: count, dtype: int64


### 📌 Preprocessing: `OBS_30_CNT_SOCIAL_CIRCLE` (Social Surroundings Observed with 30 DPD)

- **Action Taken:** Imputed 1,021 missing values using feature median (0.0) and capped extreme outliers at an upper threshold of 10.
- **Method Applied:**
  `df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'] = df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'].fillna(df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'].median())`
  `df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'] = df_main_clean['OBS_30_CNT_SOCIAL_CIRCLE'].clip(upper=10)`
- **Rationale:**
  - Resolves 1,021 missing values (0.33% missing rate).
  - Over 99% of applicants have between 0 and 6 observed contacts in their social circle.
  - An extreme outlier of 348 severe observations heavily distorts feature variance; capping at 10 neutralizes noise while preserving relative social risk rank.

In [26]:
# 1. Impute missing values with median (0.0)
def30_median = df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'].median()
df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'] = df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'].fillna(def30_median)

# 2. Cap extreme outliers at upper threshold of 5
df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'] = df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'].clip(upper=5)

# 3. Verification log
print(f"✅ DEF_30_CNT_SOCIAL_CIRCLE - Remaining Missing Values: {df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'].isnull().sum()}")
print(f"✅ DEF_30_CNT_SOCIAL_CIRCLE - Max Value after capping: {df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'].max()}")
print("✅ DEF_30_CNT_SOCIAL_CIRCLE - Value counts summary:")
print(df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'].value_counts().sort_index())

✅ DEF_30_CNT_SOCIAL_CIRCLE - Remaining Missing Values: 0
✅ DEF_30_CNT_SOCIAL_CIRCLE - Max Value after capping: 5.0
✅ DEF_30_CNT_SOCIAL_CIRCLE - Value counts summary:
DEF_30_CNT_SOCIAL_CIRCLE
0.0    272345
1.0     28328
2.0      5323
3.0      1192
4.0       253
5.0        70
Name: count, dtype: int64


### 📌 Preprocessing: `DEF_30_CNT_SOCIAL_CIRCLE` (Social Surroundings Defaulted on 30 DPD)

- **Action Taken:** Imputed 1,021 missing values using feature median (0.0) and capped extreme outliers at an upper threshold of 5.
- **Method Applied:**
  `df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'] = df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'].fillna(df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'].median())`
  `df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'] = df_main_clean['DEF_30_CNT_SOCIAL_CIRCLE'].clip(upper=5)`
- **Rationale:**
  - Resolves 1,021 missing values (0.33% missing rate), aligning perfectly with `OBS_30_CNT_SOCIAL_CIRCLE`.
  - Over 99.9% of applicants have between 0 and 3 defaulted contacts in their social circle.
  - Extreme outlier of 34 defaulted contacts distorts social default ratio calculations (`DEF_30 / OBS_30`); capping at 5 eliminates noise while preserving severe risk signal.

In [27]:
# 1. Impute missing values with median (0.0)
obs60_median = df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'].median()
df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'] = df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'].fillna(obs60_median)

# 2. Cap extreme outliers at upper threshold of 10
df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'] = df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'].clip(upper=10)

# 3. Verification log
print(f"✅ OBS_60_CNT_SOCIAL_CIRCLE - Remaining Missing Values: {df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'].isnull().sum()}")
print(f"✅ OBS_60_CNT_SOCIAL_CIRCLE - Max Value after capping: {df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'].max()}")
print("✅ OBS_60_CNT_SOCIAL_CIRCLE - Value counts summary:")
print(df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'].value_counts().sort_index())

✅ OBS_60_CNT_SOCIAL_CIRCLE - Remaining Missing Values: 0
✅ OBS_60_CNT_SOCIAL_CIRCLE - Max Value after capping: 10.0
✅ OBS_60_CNT_SOCIAL_CIRCLE - Value counts summary:
OBS_60_CNT_SOCIAL_CIRCLE
0.0     165687
1.0      48870
2.0      29766
3.0      20215
4.0      13946
5.0       9463
6.0       6349
7.0       4344
8.0       2886
9.0       1959
10.0      4026
Name: count, dtype: int64


### 📌 Preprocessing: `OBS_60_CNT_SOCIAL_CIRCLE` (Social Surroundings Observed with 60 DPD)

- **Action Taken:** Imputed 1,021 missing values using feature median (0.0) and capped extreme outliers at an upper threshold of 10.
- **Method Applied:**
  `df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'] = df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'].fillna(df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'].median())`
  `df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'] = df_main_clean['OBS_60_CNT_SOCIAL_CIRCLE'].clip(upper=10)`
- **Rationale:**
  - Resolves 1,021 missing values (0.33% missing rate), maintaining feature alignment across `SOCIAL_CIRCLE` metrics.
  - Over 99% of applicants have between 0 and 6 observed contacts in their social circle for the 60 DPD window.
  - An extreme outlier of 344 observations heavily distorts feature scale; capping at 10 eliminates noise while maintaining structural parity with 30-day features.

In [28]:
# 1. Impute missing values with median (0.0)
def60_median = df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'].median()
df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'] = df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'].fillna(def60_median)

# 2. Cap extreme outliers at upper threshold of 5
df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'] = df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'].clip(upper=5)

# 3. Verification log
print(f"✅ DEF_60_CNT_SOCIAL_CIRCLE - Remaining Missing Values: {df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'].isnull().sum()}")
print(f"✅ DEF_60_CNT_SOCIAL_CIRCLE - Max Value after capping: {df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'].max()}")
print("✅ DEF_60_CNT_SOCIAL_CIRCLE - Value counts summary:")
print(df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'].value_counts().sort_index())

✅ DEF_60_CNT_SOCIAL_CIRCLE - Remaining Missing Values: 0
✅ DEF_60_CNT_SOCIAL_CIRCLE - Max Value after capping: 5.0
✅ DEF_60_CNT_SOCIAL_CIRCLE - Value counts summary:
DEF_60_CNT_SOCIAL_CIRCLE
0.0    281742
1.0     21841
2.0      3170
3.0       598
4.0       135
5.0        25
Name: count, dtype: int64


### 📌 Preprocessing: `DEF_60_CNT_SOCIAL_CIRCLE` (Social Surroundings Defaulted on 60 DPD)

- **Action Taken:** Imputed 1,021 missing values using feature median (0.0) and capped extreme outliers at an upper threshold of 5.
- **Method Applied:**
  `df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'] = df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'].fillna(df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'].median())`
  `df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'] = df_main_clean['DEF_60_CNT_SOCIAL_CIRCLE'].clip(upper=5)`
- **Rationale:**
  - Resolves 1,021 missing values (0.33% missing rate), maintaining feature alignment across all four `SOCIAL_CIRCLE` metrics.
  - Over 99.9% of applicants have between 0 and 3 defaulted contacts in the 60 DPD window.
  - Extreme outlier of 24 defaulted contacts distorts social default ratio calculations (`DEF_60 / OBS_60`); capping at 5 eliminates extreme scale noise while preserving structural alignment with 30-day default features.
    -

In [32]:
!pip install fastparquet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 715.8/715.8 kB 4.3 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 6.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastparquet]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [33]:
import os

output_dir = '../../data/processed' if os.path.exists('../../data') else '../data/processed'
os.makedirs(output_dir, exist_ok=True)

parquet_file = os.path.join(output_dir, 'df_main_clean.parquet')

# Thêm tham số engine='fastparquet'
df_main_clean.to_parquet(parquet_file, index=False, engine='fastparquet')

file_size_mb = os.path.getsize(parquet_file) / (1024 * 1024)
print(f"✅ Đã lưu thành công dữ liệu sạch ({df_main_clean.shape[0]:,} dòng, {df_main_clean.shape[1]} cột)")
print(f"📁 Vị trí: {os.path.abspath(parquet_file)}")
print(f"💾 Dung lượng file Parquet: {file_size_mb:.2f} MB")

✅ Đã lưu thành công dữ liệu sạch (307,511 dòng, 23 cột)
📁 Vị trí: /Users/nguyenminhtri/FinalYearPro/data/processed/df_main_clean.parquet
💾 Dung lượng file Parquet: 16.39 MB
